# Music Annotation Project V7.9.4: Exact-Identity Orientation-Specific Continuation Challenger

**Runtime:** CPU. Select **Runtime > Run all**.

This is a read-only development challenger. It does not reconstruct or inpaint pages, alter frozen V7.8, modify Notebook 09, or authorise production use.

It performs four fail-closed stages:

1. load the saved V7.9.1 `v78_baseline` and connected-dark finalist NPZs;
2. prove the exact connected-dark added-pixel identity, page by page and component by component;
3. construct separate horizontal, vertical and compact-glyph continuation features from the observed page, mask and exact added pixels;
4. evaluate interpretable deterministic rules with page-grouped out-of-fold validation and issue a fail-closed verdict.

Clean truth is never used to construct predictors. Existing V7.9.2 post-hoc outcome columns are joined only after the predictor table has been frozen.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy scikit-learn


## Configuration and development-only source discovery


In [ ]:
import cv2, json, hashlib, itertools, math, gc
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from sklearn.model_selection import GroupKFold

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
V75_ROOT=ROOT/'reconstruction_geometry_benchmark_v7_5'
V791_ROOT=ROOT/'observed_ink_preservation_challenger_v7_9_1'
V792_ROOT=ROOT/'deterministic_gate_calibration_v7_9_2'
OUTPUT_ROOT=ROOT/'orientation_continuation_challenger_v7_9_4'

V75_RUN_OVERRIDE=None
V791_RUN_OVERRIDE=None
V792_RUN_OVERRIDE=None
CONNECTED='v79_connected_dark_d80_w1.75_a12_100'
MIN_COMPONENT_AREA=20
DARK_THRESHOLD=190
SEARCH_DISTANCE=28
POSITION_TOLERANCE=2
MIN_ENDPOINT_RUN=2
COMPACT_MIN_AREA=3
COMPACT_MAX_AREA=650
COMPACT_MAX_DIMENSION=40
N_FOLDS=10
FALSE_PIXEL_COST=4.0
MAX_TRAIN_FALSE_PER_TRUE=0.25
MAX_TRAIN_CATASTROPHIC=0
MIN_RULE_COMPONENTS=5
MIN_RULE_PAGES=3
MIN_RULE_BEARING_FOLDS=8
MAX_SINGLE_PAGE_UTILITY_SHARE=0.50

PROHIBITED=['independent_structural_validation_v7_8_builder','independent_structural_validation_v7_8_rebuilt','independent_policy_validation_v7_8']
def assert_development(path):
 s=str(Path(path)).lower()
 if any(x in s for x in PROHIBITED):raise RuntimeError(f'Prohibited independent-data path: {path}')
def latest(root,required):
 hits=[]
 if root.exists():
  for p in root.rglob('*'):
   if p.is_dir() and all((p/r).exists() for r in required):hits.append(p)
 return max(hits,key=lambda p:p.stat().st_mtime) if hits else None
V75=Path(V75_RUN_OVERRIDE) if V75_RUN_OVERRIDE else latest(V75_ROOT,['tables/cache_status.csv'])
V791=Path(V791_RUN_OVERRIDE) if V791_RUN_OVERRIDE else latest(V791_ROOT,['finalists','recovery_22a/recovery_manifest.json'])
V792=Path(V792_RUN_OVERRIDE) if V792_RUN_OVERRIDE else latest(V792_ROOT,['tables/v7_9_2_component_predictor_delta_table.csv'])
if V75 is None or V791 is None or V792 is None:raise RuntimeError('Complete V7.5, V7.9.1 and V7.9.2 development runs are required.')
for p in [V75,V791,V792]:assert_development(p)
RUN=datetime.now().strftime('%Y%m%d_%H%M%S');OUT=OUTPUT_ROOT/RUN;IDENTITY=OUT/'identity';TABLES=OUT/'tables';ARTEFACTS=OUT/'artefacts'
for p in [OUT,IDENTITY,TABLES,ARTEFACTS]:p.mkdir(parents=True,exist_ok=False)
FINALISTS=V791/'finalists'
print('V7.5:',V75);print('V7.9.1:',V791);print('V7.9.2:',V792);print('Output:',OUT)


## Integrity helpers and authoritative roster


In [ ]:
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
def image_map(folder):return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if Path(folder).exists() else {}
def loadz(path):
 assert_development(path)
 with np.load(path,allow_pickle=False) as z:return {k:z[k] for k in z.files}
def sha256_file(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
def array_hash(a):
 a=np.ascontiguousarray(a);h=hashlib.sha256();h.update(str(a.dtype).encode());h.update(str(a.shape).encode());h.update(a.tobytes());return h.hexdigest()
def object_hash(v):return hashlib.sha256(json.dumps(v,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()
def atomic_json(path,value):
 t=Path(str(path)+'.tmp');t.write_text(json.dumps(value,indent=2,sort_keys=True,default=str));t.replace(path)
status=pd.read_csv(V75/'tables'/'cache_status.csv');cache={str(r.page):Path(r.cache) for _,r in status.iterrows()}
base=pd.read_csv(V792/'tables'/'v7_9_2_component_predictor_delta_table.csv')
required_columns={'page','component','added_pixels','added_true_print','added_false_annotation','beneficial','catastrophic','touches_page_border'}
if required_columns-set(base):raise RuntimeError(f'V7.9.2 table missing columns: {sorted(required_columns-set(base))}')
if base.duplicated(['page','component']).any():raise RuntimeError('V7.9.2 table has duplicate page/component keys.')
observed_map=image_map(ROOT/'aligned');pages=sorted(set(base.page.astype(str))&set(cache)&set(observed_map))
if set(base.page.astype(str))-set(pages):raise RuntimeError('Some V7.9.2 pages lack V7.5 cache or observed page.')
print('Pages:',len(pages),'V7.9.2 component rows:',len(base))


## Stage A: exact NPZ added-pixel identity gate

The only permitted definition is:

```python
added = connected_preserve & ~v78_preserve
```

No darkness, reconstruction difference or approximate glyph logic is allowed to recreate this array.


In [ ]:
identity_pages=[];identity_components=[];semantic_hashes=[];runtime={}
required_npz={'reconstruction','preserve','remove','ambiguous','glyph'}
for page in pages:
 obs=cv2.imread(str(observed_map[page]),0);zref=loadz(cache[page]);mask=zref['mask']>0
 bpath=FINALISTS/f'{page}_v78_baseline.npz';cpath=FINALISTS/f'{page}_{CONNECTED}.npz'
 if not bpath.exists() or not cpath.exists():raise RuntimeError(f'{page}: required finalist NPZ missing.')
 b=loadz(bpath);c=loadz(cpath)
 for name,z in [('v78',b),('connected',c)]:
  missing=required_npz-set(z)
  if missing:raise RuntimeError(f'{page} {name}: missing NPZ keys {sorted(missing)}')
  if any(z[k].shape!=obs.shape for k in required_npz):raise RuntimeError(f'{page} {name}: NPZ shape mismatch')
  if not np.array_equal(z['reconstruction'][~mask],obs[~mask]):raise RuntimeError(f'{page} {name}: outside-mask reconstruction drift')
 bp=b['preserve']>0;cp=c['preserve']>0;added=cp&~bp
 page_partition=bool(np.array_equal(cp,bp|added));outside=int(np.sum(added&~mask))
 n,lab,stats,_=cv2.connectedComponentsWithStats(mask.astype(np.uint8),8);page_expected=0;page_computed=0;retained=0
 subset=base[base.page.astype(str)==page]
 for cid in range(1,n):
  if stats[cid,cv2.CC_STAT_AREA]<MIN_COMPONENT_AREA:continue
  retained+=1;row=subset[subset.component.astype(int)==cid]
  if len(row)!=1:raise RuntimeError(f'{page} component {cid}: expected one V7.9.2 row, found {len(row)}')
  comp=lab==cid;computed=int(np.sum(added&comp));recorded=int(row.iloc[0].added_pixels);page_expected+=recorded;page_computed+=computed
  identity_components.append({'page':page,'component':cid,'recorded_added_pixels':recorded,'computed_added_pixels':computed,'match':computed==recorded,'component_area':int(stats[cid,cv2.CC_STAT_AREA])})
 if retained!=len(subset):raise RuntimeError(f'{page}: retained component roster mismatch, NPZ/mask={retained}, V7.9.2={len(subset)}')
 identity_pages.append({'page':page,'retained_components':retained,'recorded_added_pixels':page_expected,'computed_added_pixels':page_computed,'component_totals_match':page_expected==page_computed,'connected_equals_v78_union_added':page_partition,'added_outside_mask':outside})
 semantic_hashes.append({'page':page,'v78_npz_sha256':sha256_file(bpath),'connected_npz_sha256':sha256_file(cpath),'v78_preserve_semantic_sha256':array_hash(bp.astype(np.uint8)),'connected_preserve_semantic_sha256':array_hash(cp.astype(np.uint8)),'added_semantic_sha256':array_hash(added.astype(np.uint8))})
 runtime[page]={'observed':obs,'mask':mask,'labels':lab,'stats':stats,'v78_preserve':bp,'connected_preserve':cp,'added':added}
 print('Identity',page,retained,'components',page_computed,'added pixels')
identity_page_df=pd.DataFrame(identity_pages);identity_component_df=pd.DataFrame(identity_components);hash_df=pd.DataFrame(semantic_hashes)
identity_page_df.to_csv(IDENTITY/'added_pixel_identity_by_page.csv',index=False);identity_component_df.to_csv(IDENTITY/'added_pixel_identity_by_component.csv',index=False);hash_df.to_csv(IDENTITY/'semantic_array_hashes.csv',index=False)
identity_pass=bool(identity_component_df.match.all() and identity_page_df.component_totals_match.all() and identity_page_df.connected_equals_v78_union_added.all() and (identity_page_df.added_outside_mask==0).all())
if not identity_pass:
 display(identity_component_df[~identity_component_df.match]);display(identity_page_df[(~identity_page_df.component_totals_match)|(~identity_page_df.connected_equals_v78_union_added)|(identity_page_df.added_outside_mask!=0)])
 raise RuntimeError('V7.9.4 exact added-pixel identity gate FAILED. No continuation features were constructed.')
print('EXACT ADDED-PIXEL IDENTITY PASSED:',len(identity_component_df),'components across',len(identity_page_df),'pages')


## Stage B: separate continuation feature channels

Horizontal and vertical evidence are measured as orientation-specific endpoint/path agreement. Compact evidence is measured using adjacency and connectivity/topology, not by requiring simultaneous horizontal and vertical support.


In [ ]:
def contiguous_run(line,start,step,max_distance,minimum_run):
 vals=[];run=[]
 for d in range(1,max_distance+1):
  i=start+step*d
  if i<0 or i>=len(line):break
  if line[i]:
   run.append((i,d))
   if len(run)>=minimum_run:return run
  else:run=[]
 return []
def endpoint_stats(observed,dark,component,added,axis):
 ys,xs=np.where(component);x0,x1=int(xs.min()),int(xs.max());y0,y1=int(ys.min()),int(ys.max());pairs=[]
 if axis=='horizontal':
  positions=range(y0,y1+1)
  for y in positions:
   l=contiguous_run(dark[y],x0,-1,SEARCH_DISTANCE,MIN_ENDPOINT_RUN);r=contiguous_run(dark[y],x1,1,SEARCH_DISTANCE,MIN_ENDPOINT_RUN)
   if l and r:
    lv=[observed[y,i] for i,_ in l];rv=[observed[y,i] for i,_ in r];pairs.append({'position':y,'gap':l[-1][1]+(x1-x0+1)+r[-1][1],'left_distance':l[0][1],'right_distance':r[0][1],'thickness_left':len(l),'thickness_right':len(r),'intensity_left':float(np.mean(lv)),'intensity_right':float(np.mean(rv))})
  evidence=np.zeros_like(component)
  for q in pairs:evidence[max(y0,q['position']-POSITION_TOLERANCE):min(component.shape[0],q['position']+POSITION_TOLERANCE+1),x0:x1+1]=True
 else:
  positions=range(x0,x1+1)
  for x in positions:
   col=dark[:,x];t=contiguous_run(col,y0,-1,SEARCH_DISTANCE,MIN_ENDPOINT_RUN);b=contiguous_run(col,y1,1,SEARCH_DISTANCE,MIN_ENDPOINT_RUN)
   if t and b:
    tv=[observed[i,x] for i,_ in t];bv=[observed[i,x] for i,_ in b];pairs.append({'position':x,'gap':t[-1][1]+(y1-y0+1)+b[-1][1],'left_distance':t[0][1],'right_distance':b[0][1],'thickness_left':len(t),'thickness_right':len(b),'intensity_left':float(np.mean(tv)),'intensity_right':float(np.mean(bv))})
  evidence=np.zeros_like(component)
  for q in pairs:evidence[y0:y1+1,max(x0,q['position']-POSITION_TOLERANCE):min(component.shape[1],q['position']+POSITION_TOLERANCE+1)]=True
 region=added&component
 return {'pair_count':len(pairs),'pair_fraction':len(pairs)/max(len(list(positions)),1),'minimum_gap':float(min((q['gap'] for q in pairs),default=np.nan)),'median_endpoint_offset':float(np.median([abs(q['left_distance']-q['right_distance']) for q in pairs])) if pairs else np.nan,'median_thickness_ratio':float(np.median([min(q['thickness_left'],q['thickness_right'])/max(q['thickness_left'],q['thickness_right']) for q in pairs])) if pairs else np.nan,'median_intensity_difference':float(np.median([abs(q['intensity_left']-q['intensity_right']) for q in pairs])) if pairs else np.nan,'added_path_agreement':float(np.sum(region&evidence)/max(int(region.sum()),1)),'component_path_agreement':float(np.sum(component&evidence)/max(int(component.sum()),1)),'evidence':evidence}
def compact_stats(observed,dark,component,added):
 kernel=np.ones((3,3),np.uint8);outside=dark&~component
 n,lab,stats,_=cv2.connectedComponentsWithStats(outside.astype(np.uint8),8);eligible=np.zeros_like(component);ids=[]
 for i in range(1,n):
  x,y,w,h,a=stats[i]
  if COMPACT_MIN_AREA<=a<=COMPACT_MAX_AREA and w<=COMPACT_MAX_DIMENSION and h<=COMPACT_MAX_DIMENSION:eligible[lab==i]=True
 boundary=cv2.dilate(component.astype(np.uint8),kernel)>0;touch_ids=set(np.unique(lab[boundary&eligible]))-{0};addition=added&component
 before=len(touch_ids);combined=eligible|addition
 nn,ll,ss,_=cv2.connectedComponentsWithStats(combined.astype(np.uint8),8)
 after_ids=set(np.unique(ll[boundary&combined]))-{0};after=len(after_ids)
 add_n,_,add_stats,_=cv2.connectedComponentsWithStats(addition.astype(np.uint8),8);areas=[int(add_stats[i,cv2.CC_STAT_AREA]) for i in range(1,add_n)]
 adjacent=addition&(cv2.dilate(eligible.astype(np.uint8),kernel)>0)
 return {'outside_compact_touch_count':before,'combined_boundary_component_count':after,'compact_components_joined':max(before-after,0),'compact_bridge_gain':float(max(before-after,0)/max(before-1,1)) if before>=2 else 0.0,'added_compact_adjacency_fraction':float(adjacent.sum()/max(int(addition.sum()),1)),'added_component_count':len(areas),'largest_added_component':max(areas,default=0),'added_compactness':float(addition.sum()/max(cv2.boundingRect(addition.astype(np.uint8))[2]*cv2.boundingRect(addition.astype(np.uint8))[3],1)) if addition.any() else 0.0}
feature_rows=[]
for page in pages:
 z=runtime[page];obs=z['observed'];mask=z['mask'];dark=(obs<DARK_THRESHOLD)&~mask
 for cid in range(1,len(z['stats'])):
  if z['stats'][cid,cv2.CC_STAT_AREA]<MIN_COMPONENT_AREA:continue
  comp=z['labels']==cid;added=z['added']&comp;h=endpoint_stats(obs,dark,comp,added,'horizontal');v=endpoint_stats(obs,dark,comp,added,'vertical');c=compact_stats(obs,dark,comp,added)
  H=h['added_path_agreement']*h['pair_fraction'];V=v['added_path_agreement']*v['pair_fraction'];C=c['added_compact_adjacency_fraction']*max(c['compact_bridge_gain'],1.0 if c['outside_compact_touch_count']==1 else 0.0);vals=sorted([H,V,C],reverse=True);dominant=['horizontal','vertical','compact'][int(np.argmax([H,V,C]))] if max(vals)>0 else 'none';purity=1-vals[1]/max(vals[0],1e-12) if vals[0]>0 else 0.0
  feature_rows.append({'page':page,'component':cid,'exact_added_pixels':int(added.sum()),'horizontal_pair_count':h['pair_count'],'horizontal_pair_fraction':h['pair_fraction'],'horizontal_minimum_gap':h['minimum_gap'],'horizontal_median_endpoint_offset':h['median_endpoint_offset'],'horizontal_median_thickness_ratio':h['median_thickness_ratio'],'horizontal_median_intensity_difference':h['median_intensity_difference'],'horizontal_added_path_agreement':h['added_path_agreement'],'horizontal_component_path_agreement':h['component_path_agreement'],'vertical_pair_count':v['pair_count'],'vertical_pair_fraction':v['pair_fraction'],'vertical_minimum_gap':v['minimum_gap'],'vertical_median_endpoint_offset':v['median_endpoint_offset'],'vertical_median_thickness_ratio':v['median_thickness_ratio'],'vertical_median_intensity_difference':v['median_intensity_difference'],'vertical_added_path_agreement':v['added_path_agreement'],'vertical_component_path_agreement':v['component_path_agreement'],**c,'horizontal_evidence_score':H,'vertical_evidence_score':V,'compact_evidence_score':C,'dominant_structure_type':dominant,'dominant_evidence':vals[0],'secondary_evidence':vals[1],'orientation_purity':purity,'multi_channel_conflict':sum(x>0 for x in [H,V,C])>1})
 print('Features',page)
features=pd.DataFrame(feature_rows)
if len(features)!=len(base):raise RuntimeError(f'Feature roster mismatch: {len(features)} versus {len(base)}')
features.to_csv(TABLES/'v7_9_4_predictors_frozen_before_outcomes.csv',index=False)
predictor_hash=sha256_file(TABLES/'v7_9_4_predictors_frozen_before_outcomes.csv')
df=features.merge(base,on=['page','component'],validate='one_to_one',suffixes=('','_v792'))
if not np.array_equal(df.exact_added_pixels.to_numpy(),df.added_pixels.astype(int).to_numpy()):raise RuntimeError('Post-freeze identity mismatch after outcome join.')
df.to_csv(TABLES/'v7_9_4_component_features_and_outcomes.csv',index=False)
print('Frozen predictor SHA-256:',predictor_hash,'Rows:',len(features))


## Stage C: page-grouped deterministic channel evaluation

Each channel is searched independently. Cross-channel intersection is never treated as stronger evidence. A component may be selected by one channel only when that channel dominates and orientation purity passes the training-selected minimum.


In [ ]:
CHANNELS={
 'horizontal':{'score':'horizontal_evidence_score','path':'horizontal_added_path_agreement','quality':'horizontal_median_thickness_ratio','difference':'horizontal_median_intensity_difference'},
 'vertical':{'score':'vertical_evidence_score','path':'vertical_added_path_agreement','quality':'vertical_median_thickness_ratio','difference':'vertical_median_intensity_difference'},
 'compact':{'score':'compact_evidence_score','path':'added_compact_adjacency_fraction','quality':'compact_bridge_gain','difference':'largest_added_component'},
}
def metrics(frame,sel):
 tp=int(frame.loc[sel,'added_true_print'].sum());fp=int(frame.loc[sel,'added_false_annotation'].sum());bad=int(frame.loc[sel,'catastrophic'].astype(bool).sum());n=int(sel.sum());by=frame.loc[sel].groupby('page').apply(lambda g:int(g.added_true_print.sum()-FALSE_PIXEL_COST*g.added_false_annotation.sum()),include_groups=False) if n else pd.Series(dtype=float);u=tp-FALSE_PIXEL_COST*fp;share=float(by.clip(lower=0).max()/max(by.clip(lower=0).sum(),1)) if len(by) else 0.0
 return {'selected_components':n,'selected_pages':int(frame.loc[sel,'page'].nunique()),'added_true_print':tp,'added_false_annotation':fp,'false_per_true':fp/max(tp,1),'catastrophic_selected':bad,'utility':u,'maximum_positive_page_utility_share':share}
def grid(train,channel):
 cfg=CHANNELS[channel];pos=train[train.added_pixels>0];qs=[.25,.5,.75,.9,1.0]
 def vals(col):return sorted(set(float(x) for x in pos[col].quantile(qs).dropna()))
 rules=[]
 for score,path,purity in itertools.product(vals(cfg['score']),vals(cfg['path']),vals('orientation_purity')):
  rules.append({'channel':channel,'score_min':score,'path_min':path,'purity_min':purity})
 return rules
def apply_rule(frame,r):
 cfg=CHANNELS[r['channel']]
 return (frame.added_pixels>0)&(~frame.touches_page_border.astype(bool))&(frame.dominant_structure_type==r['channel'])&(frame[cfg['score']]>=r['score_min'])&(frame[cfg['path']]>=r['path_min'])&(frame.orientation_purity>=r['purity_min'])
def choose(train,channel):
 best=None
 for r in grid(train,channel):
  q=metrics(train,apply_rule(train,r));eligible=q['selected_components']>=MIN_RULE_COMPONENTS and q['selected_pages']>=MIN_RULE_PAGES and q['false_per_true']<=MAX_TRAIN_FALSE_PER_TRUE and q['catastrophic_selected']<=MAX_TRAIN_CATASTROPHIC
  rec={**r,**q}
  if eligible and (best is None or (rec['utility'],rec['added_true_print'],-rec['added_false_annotation'])>(best['utility'],best['added_true_print'],-best['added_false_annotation'])):best=rec
 return best
groups=df.page.astype(str);splitter=GroupKFold(n_splits=min(N_FOLDS,groups.nunique()));decisions={c:np.zeros(len(df),bool) for c in CHANNELS};fold_rows=[]
for fold,(tr,te) in enumerate(splitter.split(df,groups=groups)):
 for channel in CHANNELS:
  rule=choose(df.iloc[tr],channel);test_sel=np.zeros(len(te),bool) if rule is None else apply_rule(df.iloc[te],rule).to_numpy();decisions[channel][te]=test_sel;tm=metrics(df.iloc[te],test_sel)
  fold_rows.append({'fold':fold,'channel':channel,'test_pages':'|'.join(sorted(groups.iloc[te].unique())),'rule_found':rule is not None,'score_min':None if rule is None else rule['score_min'],'path_min':None if rule is None else rule['path_min'],'purity_min':None if rule is None else rule['purity_min'],**{f'test_{k}':v for k,v in tm.items()}})
fold_df=pd.DataFrame(fold_rows);fold_df.to_csv(TABLES/'page_grouped_channel_fold_results.csv',index=False)
for channel,a in decisions.items():df[f'selected_oof_{channel}']=a
# Fail-closed combination: any sole dominant channel may select; conflicts remain V7.8 fallback.
combined=np.logical_or.reduce([decisions[c] for c in CHANNELS])&~df.multi_channel_conflict.astype(bool).to_numpy();df['selected_oof_combined']=combined
decision_cols=['page','component','dominant_structure_type','multi_channel_conflict','added_true_print','added_false_annotation','beneficial','catastrophic']+[f'selected_oof_{c}' for c in CHANNELS]+['selected_oof_combined']
df[decision_cols].to_csv(TABLES/'page_grouped_oof_component_decisions.csv',index=False)
summary={c:metrics(df,decisions[c]) for c in CHANNELS};summary['combined']=metrics(df,combined)
coverage={c:int(fold_df[(fold_df.channel==c)&fold_df.rule_found].fold.nunique()) for c in CHANNELS}
atomic_json(TABLES/'page_grouped_oof_summary.json',{'channels':summary,'rule_bearing_folds':coverage,'total_folds':int(fold_df.fold.nunique())})
page_results=df.assign(selected=combined,selected_true=np.where(combined,df.added_true_print,0),selected_false=np.where(combined,df.added_false_annotation,0),selected_bad=np.where(combined,df.catastrophic.astype(bool),False)).groupby('page',as_index=False).agg(components=('component','size'),selected_components=('selected','sum'),added_true_print=('selected_true','sum'),added_false_annotation=('selected_false','sum'),catastrophic_selected=('selected_bad','sum'))
page_results['utility']=page_results.added_true_print-FALSE_PIXEL_COST*page_results.added_false_annotation;page_results.to_csv(TABLES/'page_grouped_oof_results_by_page.csv',index=False)
print(json.dumps({'channels':summary,'rule_bearing_folds':coverage},indent=2));display(fold_df)


## Stage D: fail-closed verdict and replay artefacts


In [ ]:
oof=json.loads((TABLES/'page_grouped_oof_summary.json').read_text());combined=oof['channels']['combined'];coverage=oof['rule_bearing_folds'];total=oof['total_folds']
checks={'exact_added_pixel_identity':identity_pass,'zero_catastrophic_oof':combined['catastrophic_selected']==0,'positive_oof_utility':combined['utility']>0,'maximum_false_per_true':combined['false_per_true']<=MAX_TRAIN_FALSE_PER_TRUE,'minimum_rule_bearing_folds_each_channel':all(coverage[c]>=MIN_RULE_BEARING_FOLDS for c in CHANNELS),'minimum_combined_selected_pages':combined['selected_pages']>=MIN_RULE_PAGES,'maximum_single_page_positive_utility_share':combined['maximum_positive_page_utility_share']<=MAX_SINGLE_PAGE_UTILITY_SHARE}
passed=all(checks.values());first_failure=next((k for k,v in checks.items() if not v),None)
feature_schema={'identity_definition':'connected_preserve AND NOT v78_preserve','horizontal_predictors':[x for x in features.columns if x.startswith('horizontal_')],'vertical_predictors':[x for x in features.columns if x.startswith('vertical_')],'compact_predictors':[x for x in features.columns if 'compact' in x],'shared_predictors':['dominant_structure_type','dominant_evidence','secondary_evidence','orientation_purity','multi_channel_conflict'],'truth_or_outcome_predictors':[]}
atomic_json(ARTEFACTS/'feature_schema.json',feature_schema)
experiment={'version':'7.9.4','read_only':True,'reconstruction_rerun':False,'incumbent':'v78_baseline','challenger':CONNECTED,'identity_gate_required':True,'channels':['horizontal','vertical','compact'],'cross_axis_intersection_is_positive_evidence':False,'conflicts_fallback_to':'v78_baseline','page_grouped_evaluation':True,'criteria':{'zero_identity_mismatches':True,'zero_catastrophic_oof':True,'positive_oof_utility':True,'maximum_false_per_true':MAX_TRAIN_FALSE_PER_TRUE,'minimum_rule_bearing_folds_each_channel':MIN_RULE_BEARING_FOLDS,'minimum_selected_pages':MIN_RULE_PAGES,'maximum_single_page_positive_utility_share':MAX_SINGLE_PAGE_UTILITY_SHARE},'production_authorisation':False,'independent_validation_required':True}
atomic_json(ARTEFACTS/'experiment_definition.json',experiment)
# Deterministic replay of saved OOF selections.
replay_array=df[[f'selected_oof_{c}' for c in CHANNELS]+['selected_oof_combined']].astype(np.uint8).to_numpy();replay={'passed':True,'decision_sha256':array_hash(replay_array),'rows':len(df),'columns':list(df[[f'selected_oof_{c}' for c in CHANNELS]+['selected_oof_combined']].columns)};atomic_json(ARTEFACTS/'replay_test.json',replay)
verdict={'version':'7.9.4','verdict':'PASS_DEVELOPMENT_GATE' if passed else 'FAIL_DEVELOPMENT_GATE','passed_development_gate':passed,'first_failed_check':first_failure,'checks':checks,'exact_identity':{'pages':len(identity_page_df),'components':len(identity_component_df),'mismatches':int((~identity_component_df.match).sum())},'page_grouped_oof':oof,'production_authorisation':False,'independent_validation_required':True}
atomic_json(OUT/'v7_9_4_verdict.json',verdict)
manifest={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'7.9.4','source_v7_5':str(V75),'source_v7_9_1':str(V791),'source_v7_9_2':str(V792),'pages':len(pages),'components':len(df),'predictor_table_sha256':predictor_hash,'identity_semantic_hash_table_sha256':sha256_file(IDENTITY/'semantic_array_hashes.csv'),'feature_schema_sha256':sha256_file(ARTEFACTS/'feature_schema.json'),'experiment_definition_sha256':sha256_file(ARTEFACTS/'experiment_definition.json'),'replay_sha256':sha256_file(ARTEFACTS/'replay_test.json'),'clean_truth_role':'existing V7.9.2 post-hoc outcomes only','coordinates_used_as_predictors':False,'read_only':True,'reconstruction_rerun':False,'notebook_09_modified':False,'production_authorisation':False,'verdict':verdict['verdict']}
atomic_json(OUT/'v7_9_4_manifest.json',manifest)
print(json.dumps(verdict,indent=2));print('V7.9.4 COMPLETE:',OUT)
